# Classification Partie 1 - Demandes de prêt immobilier aux États-Unis

## Import des librairies :

In [22]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier 
from scipy.stats import loguniform
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from autogluon.tabular import TabularPredictor

## Pré-traitement sur les données :

##### Chargement des données d'entraînement et de test :

In [9]:
data_train = pd.read_csv("train.csv")
data_test = pd.read_csv("test.csv")

In [ ]:
# Nombre de valeurs manquantes par colonne (train puis test) :
print(data_train.isna().sum())
print(data_test.isna().sum())

Il y a des valeurs manquantes (NaN) aussi bien dans le dataset d'entraînement (`data_train`) que dans le dataset de test (`data_test`).

### Variables continues :

Les variables continues sont : `derived_msa-md`, `loan_amount`, `loan_to_value_ratio`, `loan_term`, `property_value`, `income`, `tract_population`, `tract_minority_population_percent`, `ffiec_msa_md_median_family_income`, `tract_to_msa_income_percentage`, `tract_owner_occupied_units`, `tract_one_to_four_family_homes`, `tract_median_age_of_housing_units`, `county_code`

On doit remplir les valeurs manquantes pour les variables suivantes : `loan_to_value_ratio`, `property_value`, `income`, `loan_term`, `county_code`. 

D'abord, on crée un indicateur de valeur manquante : on produit une nouvelle colonne `col_missing` qui vaut 1 si la valeur d’origine était absente et 0 sinon. Cette colonne conserve l’information « la valeur était manquante » avant qu’elle soit effacée à l’étape suivante.

Ensuite, on impute par la médiane : on remplace chaque valeur manquante par la médiane de la colonne de `data_train`.

In [10]:
num_cols = ["income", "property_value", "loan_term", "loan_to_value_ratio", "county_code"]

for col in num_cols:
    data_train[col + "_missing"] = data_train[col].isnull().astype(int)  # 1) création d'une nouvelle colonne : indicateur de valeur manquante
    data_train[col] = data_train[col].fillna(data_train[col].median())   # 2) imputation par la médiane

for col in num_cols:
    data_test[col + "_missing"] = data_test[col].isnull().astype(int) 
    data_test[col] = data_test[col].fillna(data_train[col].median())   # médiane sur data_train pour éviter la fuite de données

### Variables catégorielles :

Il y a des variables ordinales, pour lesquelles l'ordre est important : `applicant_age`, `co-applicant_age`, `debt_to_income_ratio`

Il y a des variables nominales (pas forcément textuelles !), pour lesquelles l'ordre ne compte pas : `state_code`, `conforming_loan_limit`, `derived_loan_product_type`, `derived_dwelling_category`,  `loan_type`, `lien_status`, `preapproval`, `reverse_mortgage`, `open-end_line_of_credit`, `business_or_commercial_purpose`, `negative_amortization`, `interest_only_payment`, `balloon_payment`, `other_nonamortizing_features`, `construction_method`, `occupancy_type`, `manufactured_home_secured_property_type`, `manufactured_home_land_property_interest`, `total_units`, `applicant_credit_score_type`, `co-applicant_credit_score_type`, `derived_ethnicity`, `derived_race`, `derived_sex`, `applicant_ethnicity-1`, `co-applicant_ethnicity-1`, `applicant_ethnicity_observed`, `co-applicant_ethnicity_observed`, `applicant_race-1`, `co-applicant_race-1`, `applicant_race_observed`, `co-applicant_race_observed`, `applicant_sex`, `co-applicant_sex`, `applicant_sex_observed`, `co-applicant_sex_observed`, `submission_of_application`, `initially_payable_to_institution`, `aus-1`

Ces deux types de variables ne se traitent pas avec le même encodeur. On sépare les deux groupes :

##### Variables catégorielles ordinales :

Pour `applicant_age` et `co-applicant_age` :

In [11]:
# Chaque tranche est remplacée par son milieu (ex: "25-34" -> 30) pour garder l'ordre.
ordre_age = {"<25": 25, "25-34": 30, "35-44": 40, "45-54": 50,
         "55-64": 60, "65-74": 70, ">74": 74}

data_train["applicant_age"] = data_train["applicant_age"].map(ordre_age)
data_train["co-applicant_age"] = data_train["co-applicant_age"].map(ordre_age)
data_test["applicant_age"] = data_test["applicant_age"].map(ordre_age)
data_test["co-applicant_age"] = data_test["co-applicant_age"].map(ordre_age)

Pour `debt_to_income_ratio` :

In [12]:
ordre_debt = {"<20%": 20, "20%-<30%": 25, "30%-<36%": 33, "36": 36, "37": 37, "38": 38,
              "39": 39, "40": 40, "41": 41, "42": 42, "43": 43, "44": 44, "45": 45, "46": 46,
              "47": 47, "48": 48, "49": 49, "50%-60%": 55, ">60%": 60}

data_train["debt_to_income_ratio"] = data_train["debt_to_income_ratio"].map(ordre_debt)
data_test["debt_to_income_ratio"] = data_test["debt_to_income_ratio"].map(ordre_debt)

In [13]:
# On attribue un code (-2) aux valeurs manquantes (NaN) 
ord_cols = ["applicant_age", "co-applicant_age", "debt_to_income_ratio"]

for df in (data_train, data_test):
    df[ord_cols] = df[ord_cols].fillna(-2).astype(int)

##### Variables catégorielles nominales :

In [14]:
cols_nom = [
    "state_code", "conforming_loan_limit",
    "derived_loan_product_type", "derived_dwelling_category",
    "loan_type", "lien_status", "preapproval", "reverse_mortgage",
    "open-end_line_of_credit", "business_or_commercial_purpose",
    "negative_amortization", "interest_only_payment", "balloon_payment",
    "other_nonamortizing_features", "construction_method", "occupancy_type",
    "manufactured_home_secured_property_type",
    "manufactured_home_land_property_interest", "total_units",
    "applicant_credit_score_type", "co-applicant_credit_score_type",
    "derived_ethnicity", "derived_race", "derived_sex",
    "applicant_ethnicity-1", "co-applicant_ethnicity-1",
    "applicant_ethnicity_observed", "co-applicant_ethnicity_observed",
    "applicant_race-1", "co-applicant_race-1",
    "applicant_race_observed", "co-applicant_race_observed",
    "applicant_sex", "co-applicant_sex",
    "applicant_sex_observed", "co-applicant_sex_observed",
    "submission_of_application", "initially_payable_to_institution",
    "aus-1",
]

enc_nom = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-2) 
# unknown_value=-1 : catégorie vue dans le test mais pas dans le train
# encoded_missing_value=-2 : vraie valeur manquante (NaN)
# fit sur le train uniquement, puis transform sur le test (évite la fuite de données)
data_train[cols_nom] = enc_nom.fit_transform(data_train[cols_nom])
data_test[cols_nom] = enc_nom.transform(data_test[cols_nom])

On enregistre ces nouveaux datasets :

In [15]:
data_train.to_csv("data_train_clean.csv", index=True)
data_test.to_csv("data_test_clean.csv", index=True)

## Modèles prédictifs :

##### Chargements des données d'entraînement et de test nettoyées :

In [16]:
data_train = pd.read_csv("data_train_clean.csv")
data_test = pd.read_csv("data_test_clean.csv")

data_train = data_train.drop(columns=["Unnamed: 0"])
data_test = data_test.drop(columns=["Unnamed: 0"])

##### On sépare les covariables de la variable cible dans les données d'entraînement :

In [17]:
X_train = data_train.drop(columns=["accepted", "id"])
Y_train = data_train["accepted"]
X_test = data_test.drop(columns=["id"])

On normalise :

In [18]:
scaler = StandardScaler()
X_train_normed = scaler.fit_transform(X_train)
X_test_normed = scaler.transform(X_test)

# Nécessaire pour la régression logistique et SVM
# Inutile pour les modèles à base d'arbres

### Régression logistique :

##### Entraînement :

In [19]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_normed, Y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

##### Prédictions :

In [20]:
y_proba = model.predict_proba(X_test_normed)[:, 1] # Probabilité de la classe 1 (accepté)

submission = pd.DataFrame({
    "id": data_test["id"],
    "accepted": y_proba
})
submission.to_csv("classification_submission_logistic_regression.csv", index=False)

### Arbre de décision :

##### Recherche d'hyperparamètres et entraînement :

Optimisation de `max_depth` et `min_samples_leaf` :

In [24]:
params = {
    "max_depth": [8, 10, 12, 14, 16, 20, None],
    "min_samples_leaf": [1, 5, 10, 20, 50, 100],
}

search_tree = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    params, cv=5,
    scoring="roc_auc",
    n_jobs=-1,
)
search_tree.fit(X_train, Y_train)

print("Meilleurs paramètres :", search_tree.best_params_)
print(f"AUC : {search_tree.best_score_:.2f}")

Meilleurs paramètres : {'max_depth': 14, 'min_samples_leaf': 100}
AUC : -0.89


##### Prédictions :

In [25]:
best_tree = search_tree.best_estimator_ # modèle entraîné sur data_train avec les meilleurs paramètres
y_proba = best_tree.predict_proba(X_test)[:, 1] # Probabilité de la classe 1 (accepté)

submission = pd.DataFrame({
    "id": data_test["id"],
    "accepted": y_proba
})
submission.to_csv("classification_submission_decision_tree.csv", index=False)

### Gradient Boosting :

##### Recherche d'hyperparamètres et entraînement :

Optimisation de `max_iter`, `learning_rate`, `max_depth`, `max_leaf_nodes`, `min_samples_leaf` et `l2_regularization` :

In [ ]:
params = {
    "max_iter": [200, 400, 600, 1000],
    "learning_rate": [0.02, 0.05, 0.1],
    "max_depth": [None, 4, 6, 8],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [10, 20, 50, 100],
    "l2_regularization": [0, 0.1, 1, 10],
}

search_hgbc = RandomizedSearchCV(
    HistGradientBoostingClassifier(random_state=42),
    params, n_iter=20,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1, random_state=42,
)
search_hgbc.fit(X_train, Y_train)

print("Meilleurs paramètres :", search_hgbc.best_params_)
print(f"AUC CV : {search_hgbc.best_score_:.4f}")

##### Prédictions :

In [ ]:
best_hgbc = search_hgbc.best_estimator_
y_proba = best_hgbc.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": data_test["id"],
    "accepted": y_proba
})

submission.to_csv("classification_submission_gradient_boosting.csv", index=False)

### SVM :

##### Recherche d'hyperparamètres :

Optimisation de `C` et `Gamma` :

In [ ]:
# Le SVM est trop lent sur tout le jeu de données : la recherche d'hyperparamètres
# se fait sur un sous-échantillon de 15000 lignes :
X_sub, _, y_sub, _ = train_test_split(
    X_train, Y_train, train_size=15_000, stratify=Y_train, random_state=0
)

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", class_weight="balanced", cache_size=1000)),
])

param_dist = {
    "svm__C":     loguniform(0.1, 100),
    "svm__gamma": loguniform(1e-4, 1),
}

search_svm = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=20, cv=3, scoring="roc_auc",
    n_jobs=-1, random_state=42, verbose=2,
)
search_svm.fit(X_sub, y_sub)

print("Meilleur AUC CV :", search_svm.best_score_)
print("Meilleurs paramètres :", search_svm.best_params_)

##### Entraînement :

In [ ]:
from cuml.svm import SVC # utilisation de GPU

# C et Gamma sont copié collé de la recherche précedente
model_svm = SVC(kernel="rbf", C=0.8179499475211672, gamma=0.012561043700013555, class_weight="balanced", probability=True)
model_svm.fit(X_train_normed, Y_train)

##### Prédictions :

In [ ]:
y_proba = model_svm.predict_proba(X_test_normed)[:, 1] # Probabilité de la classe 1 (accepté)

submission = pd.DataFrame({
    "id": data_test["id"],
    "accepted": y_proba
})
submission.to_csv("classification_submission_svm.csv", index=False)

### AutoGluon :

In [ ]:
predictor = TabularPredictor(
    label="accepted",
    eval_metric="roc_auc",       
).fit(
    data_train.drop(columns=["id"]),
    presets="best",
    time_limit=300,             
    num_gpus=1,                  
)

##### Prédictions :

In [ ]:
y_proba = predictor.predict_proba(data_test.drop(columns=["id"]))
submission = pd.DataFrame({
    "id": data_test["id"],
    "accepted": y_proba.iloc[:, 1]     
})
submission.to_csv("classification_submission_auto_gluon.csv", index=False)